In [1]:
!apt-get update -qq
!apt-get install -y rabbitmq-server
!service rabbitmq-server start

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 logrotate
Suggested packages:
  erlang erlang-manpages erlang-doc lksctp-tools bsd-mailx | mailx
The following NEW packages will be installed:
  erlang-asn1 erlang-base erlang-crypto erlang-eldap erlang-ftp erlang-inets
  erlang-mnesia erlang-os-mon erlang-parsetools erlang-public-key
  erlang-runtime-tools erlang-snmp erlang-ssl erlang-syntax-tools erlang-tftp
  erlang-tools erlang-xmerl libsctp1 lo

In [2]:
!rabbitmqctl status

Status of node rabbit@a86068fed000 ...
Runtime

OS PID: 7350
OS: Linux
Uptime (seconds): 29
Is under maintenance?: false
RabbitMQ version: 3.12.1
RabbitMQ release series support status: supported
Node name: rabbit@a86068fed000
Erlang configuration: Erlang/OTP 25 [erts-13.2.2.5] [source] [64-bit] [smp:2:2] [ds:2:2:10] [async-threads:1] [jit:ns]
Crypto library: OpenSSL 3.0.13 30 Jan 2024
Erlang processes: 272 used, 1048576 limit
Scheduler run queue: 1
Cluster heartbeat timeout (net_ticktime): 60

Plugins

Enabled plugin file: /etc/rabbitmq/enabled_plugins
Enabled plugins:


Data directory

Node data directory: /var/lib/rabbitmq/mnesia/rabbit@a86068fed000
Raft data directory: /var/lib/rabbitmq/mnesia/rabbit@a86068fed000/quorum/rabbit@a86068fed000

Config files


Log file(s)

 * /var/log/rabbitmq/rabbit@a86068fed000.log
 * <stdout>

Alarms

(none)

Memory

Total memory used: 0.1306 gb
Calculation strategy: rss
Memory high watermark setting: 0.4 of available memory, computed to: 5.4423 gb



In [3]:
%cd /content
!mkdir -p stream
%cd /content/stream

/content
/content/stream


In [4]:
%%writefile dispatcher.py
import argparse
import json
import random
import time
import uuid
from datetime import datetime, timezone

import pika

EXCHANGE = "rapido-dispatch"

ZONES = {
    "bangalore": ["koramangala", "indiranagar", "whitefield"],
    "chennai": ["tnagar", "adyar", "annanagar"],
}

TYPES = ["ride", "food"]


def connect():
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()
    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True,
    )
    return connection, channel


def publish_request(channel, city=None, zone=None, request_type=None):
    city = city or random.choice(list(ZONES.keys()))
    zone = zone or random.choice(ZONES[city])
    request_type = request_type or random.choice(TYPES)

    routing_key = f"{city}.{zone}.{request_type}"

    message = {
        "id": str(uuid.uuid4())[:8],
        "city": city,
        "zone": zone,
        "type": request_type,
        "ts": datetime.now(timezone.utc).isoformat(),
    }

    channel.basic_publish(
        exchange=EXCHANGE,
        routing_key=routing_key,
        body=json.dumps(message),
        properties=pika.BasicProperties(
            content_type="application/json",
            delivery_mode=2,
        ),
    )

    print(
        f"Published | key={routing_key} | message={message}",
        flush=True,
    )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--demo-key",
        help="Publish one specific routing key before random requests",
    )
    args = parser.parse_args()

    connection, channel = connect()

    try:
        if args.demo_key:
            parts = args.demo_key.lower().split(".")
            if len(parts) != 3:
                raise ValueError("Demo key must contain exactly 3 words")

            city, zone, request_type = parts
            if city not in ZONES or zone not in ZONES[city]:
                raise ValueError("Unknown city or zone in demo key")
            if request_type not in TYPES:
                raise ValueError("Type must be ride or food")

            publish_request(channel, city, zone, request_type)

        print("Dispatcher started. Publishing every 2 seconds...", flush=True)

        while True:
            publish_request(channel)
            time.sleep(2)

    except KeyboardInterrupt:
        print("Dispatcher stopped.", flush=True)
    finally:
        connection.close()


if __name__ == "__main__":
    main()

Writing dispatcher.py


In [5]:
%%writefile driver.py
import json
import sys

import pika

EXCHANGE = "rapido-dispatch"


def main():
    if len(sys.argv) != 3:
        print("Usage: python driver.py NAME PATTERN")
        sys.exit(1)

    name = sys.argv[1]
    pattern = sys.argv[2]

    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True,
    )

    result = channel.queue_declare(
        queue="",
        exclusive=True,
        auto_delete=True,
    )
    queue_name = result.method.queue

    channel.queue_bind(
        exchange=EXCHANGE,
        queue=queue_name,
        routing_key=pattern,
    )

    channel.basic_qos(prefetch_count=1)

    print(
        f"Driver {name} listening | pattern={pattern} | queue={queue_name}",
        flush=True,
    )

    def callback(ch, method, properties, body):
        message = json.loads(body.decode())

        print(
            f"Driver={name} | key={method.routing_key} | "
            f"request={message}",
            flush=True,
        )

        ch.basic_ack(delivery_tag=method.delivery_tag)

    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback,
        auto_ack=False,
    )

    try:
        channel.start_consuming()
    except KeyboardInterrupt:
        print(f"Driver {name} stopped.", flush=True)
        channel.stop_consuming()
    finally:
        connection.close()


if __name__ == "__main__":
    main()

Writing driver.py


In [6]:
!pip install pika

In [7]:
%cd /content/stream

!nohup python -u driver.py Ravi 'bangalore.koramangala.*' > ravi.log 2>&1 &
!nohup python -u driver.py Priya 'bangalore.indiranagar.*' > priya.log 2>&1 &
!nohup python -u driver.py Arjun 'bangalore.*.ride' > arjun.log 2>&1 &
!nohup python -u driver.py Meena 'bangalore.*.food' > meena.log 2>&1 &
!nohup python -u driver.py Karthik 'chennai.tnagar.food' > karthik.log 2>&1 &
!nohup python -u driver.py Control 'bangalore.#' > control.log 2>&1 &

!sleep 5
!cat ravi.log priya.log arjun.log meena.log karthik.log control.log

/content/stream
Driver Ravi listening | pattern=bangalore.koramangala.* | queue=amq.gen-q-EmJmNGVs3WhXG3tVxrMg
Driver Priya listening | pattern=bangalore.indiranagar.* | queue=amq.gen-Y_uKnZRvuesuULOVwHMu0A
Driver Arjun listening | pattern=bangalore.*.ride | queue=amq.gen-l4JvF2gor74whqaQVgluAA
Driver Meena listening | pattern=bangalore.*.food | queue=amq.gen-iqgyARmYigWDM_H9GRn8WA
Driver Karthik listening | pattern=chennai.tnagar.food | queue=amq.gen-huZbyVEv3Yj6Zr3aEXhujQ
Driver Control listening | pattern=bangalore.# | queue=amq.gen-XZXhUZGPfMIwMwfxo8Xn6A


In [8]:
!nohup python -u dispatcher.py --demo-key bangalore.koramangala.food > dispatcher.log 2>&1 &

In [9]:
import time
time.sleep(120)

In [10]:
print("=== DISPATCHER ===")
!head -5 dispatcher.log

print("\n=== RAVI ===")
!cat ravi.log

print("\n=== MEENA ===")
!cat meena.log

print("\n=== CONTROL ===")
!cat control.log

print("\n=== PRIYA ===")
!cat priya.log

print("\n=== ARJUN ===")
!cat arjun.log

print("\n=== KARTHIK ===")
!cat karthik.log

=== DISPATCHER ===
Published | key=bangalore.koramangala.food | message={'id': 'e6df9358', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-16T13:47:05.852528+00:00'}
Dispatcher started. Publishing every 2 seconds...
Published | key=bangalore.indiranagar.ride | message={'id': 'dda824b4', 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'ride', 'ts': '2026-11-16T13:47:05.853029+00:00'}
Published | key=bangalore.whitefield.ride | message={'id': '17bf9b4b', 'city': 'bangalore', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-16T13:47:07.853618+00:00'}
Published | key=bangalore.koramangala.ride | message={'id': '53c304eb', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'ride', 'ts': '2026-11-16T13:47:09.854506+00:00'}

=== RAVI ===
Driver Ravi listening | pattern=bangalore.koramangala.* | queue=amq.gen-q-EmJmNGVs3WhXG3tVxrMg
Driver=Ravi | key=bangalore.koramangala.food | request={'id': 'e6df9358', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'f

In [11]:
!pkill -f "python -u dispatcher.py" || true

^C


D3 start


In [12]:
%%writefile /content/stream/dispatcher.py
import json
import random
import time
import uuid
from datetime import datetime, timezone

import pika

EXCHANGE = "rapido-dispatch"

ZONES = {
    "bangalore": ["koramangala", "indiranagar", "whitefield"],
    "chennai": ["tnagar", "adyar", "annanagar"],
}

TYPES = ["ride", "food"]


def connect():
    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True,
    )

    return connection, channel


def publish_request(channel, count):
    city = random.choice(list(ZONES.keys()))
    zone = random.choice(ZONES[city])
    request_type = random.choice(TYPES)

    routing_key = f"{city}.{zone}.{request_type}"

    # Every 10th message is deliberately bad.
    if count % 10 == 0:
        message = {
            "id": str(uuid.uuid4())[:8],
            "city": city,
            # Deliberately missing "zone"
            "type": request_type,
            "ts": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"BAD MESSAGE #{count} | key={routing_key} | "
            f"missing zone | id={message['id']}",
            flush=True,
        )

    else:
        message = {
            "id": str(uuid.uuid4())[:8],
            "city": city,
            "zone": zone,
            "type": request_type,
            "ts": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"GOOD MESSAGE #{count} | key={routing_key} | "
            f"id={message['id']}",
            flush=True,
        )

    channel.basic_publish(
        exchange=EXCHANGE,
        routing_key=routing_key,
        body=json.dumps(message),
        properties=pika.BasicProperties(
            content_type="application/json",
            delivery_mode=2,
        ),
    )


def main():
    connection, channel = connect()
    count = 0

    print("Dispatcher started...", flush=True)

    try:
        while True:
            count += 1
            publish_request(channel, count)
            time.sleep(2)

    except KeyboardInterrupt:
        print("Dispatcher stopped.", flush=True)

    finally:
        connection.close()


if __name__ == "__main__":
    main()

Overwriting /content/stream/dispatcher.py


In [13]:
%%writefile /content/stream/driver.py
import json
import sys

import pika

EXCHANGE = "rapido-dispatch"


def main():
    if len(sys.argv) != 3:
        print("Usage: python driver.py NAME PATTERN")
        sys.exit(1)

    name = sys.argv[1]
    pattern = sys.argv[2]

    connection = pika.BlockingConnection(
        pika.ConnectionParameters(host="localhost")
    )
    channel = connection.channel()

    channel.exchange_declare(
        exchange=EXCHANGE,
        exchange_type="topic",
        durable=True,
    )

    result = channel.queue_declare(
        queue="",
        exclusive=True,
        auto_delete=True,
    )

    queue_name = result.method.queue

    channel.queue_bind(
        exchange=EXCHANGE,
        queue=queue_name,
        routing_key=pattern,
    )

    channel.basic_qos(prefetch_count=1)

    print(
        f"Driver {name} listening | pattern={pattern}",
        flush=True,
    )

    def callback(ch, method, properties, body):
        try:
            message = json.loads(body.decode())

            # Reject messages that do not contain the zone field.
            if "zone" not in message:
                print(
                    f"REJECTED by {name} | key={method.routing_key} | "
                    f"reason=missing zone | id={message.get('id')}",
                    flush=True,
                )

                ch.basic_nack(
                    delivery_tag=method.delivery_tag,
                    requeue=False,
                )
                return

            # Process valid messages.
            print(
                f"PROCESSED by {name} | key={method.routing_key} | "
                f"message={message}",
                flush=True,
            )

            ch.basic_ack(
                delivery_tag=method.delivery_tag
            )

        except (json.JSONDecodeError, UnicodeDecodeError) as error:
            print(
                f"REJECTED by {name} | invalid JSON | {error}",
                flush=True,
            )

            ch.basic_nack(
                delivery_tag=method.delivery_tag,
                requeue=False,
            )

    channel.basic_consume(
        queue=queue_name,
        on_message_callback=callback,
        auto_ack=False,
    )

    try:
        channel.start_consuming()

    except KeyboardInterrupt:
        print(f"Driver {name} stopped.", flush=True)
        channel.stop_consuming()

    finally:
        connection.close()


if __name__ == "__main__":
    main()

Overwriting /content/stream/driver.py


In [14]:
%cd /content/stream

!pkill -f "python -u driver.py" || true

/content/stream
^C


In [15]:
!nohup python -u driver.py Ravi 'bangalore.koramangala.*' > ravi_d3.log 2>&1 &
!nohup python -u driver.py Priya 'bangalore.indiranagar.*' > priya_d3.log 2>&1 &
!nohup python -u driver.py Arjun 'bangalore.*.ride' > arjun_d3.log 2>&1 &
!nohup python -u driver.py Meena 'bangalore.*.food' > meena_d3.log 2>&1 &
!nohup python -u driver.py Karthik 'chennai.tnagar.food' > karthik_d3.log 2>&1 &
!nohup python -u driver.py Control 'bangalore.#' > control_d3.log 2>&1 &

!sleep 5
!cat ravi_d3.log priya_d3.log arjun_d3.log meena_d3.log karthik_d3.log control_d3.log

Driver Ravi listening | pattern=bangalore.koramangala.*
Driver Priya listening | pattern=bangalore.indiranagar.*
PROCESSED by Priya | key=bangalore.indiranagar.ride | message={'id': '6c17c7f0', 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'ride', 'ts': '2026-11-16T13:54:38.045152+00:00'}
Driver Arjun listening | pattern=bangalore.*.ride
PROCESSED by Arjun | key=bangalore.indiranagar.ride | message={'id': '6c17c7f0', 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'ride', 'ts': '2026-11-16T13:54:38.045152+00:00'}
Driver Meena listening | pattern=bangalore.*.food
Driver Karthik listening | pattern=chennai.tnagar.food
Driver Control listening | pattern=bangalore.#
PROCESSED by Control | key=bangalore.indiranagar.ride | message={'id': '6c17c7f0', 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'ride', 'ts': '2026-11-16T13:54:38.045152+00:00'}


In [16]:
!nohup python -u dispatcher.py > dispatcher_d3.log 2>&1 &

In [17]:
import time
time.sleep(120)

In [18]:
!pkill -f "python -u dispatcher.py" || true

^C


In [19]:
!cat dispatcher_d3.log

Dispatcher started...
GOOD MESSAGE #1 | key=chennai.tnagar.ride | id=957bae40
GOOD MESSAGE #2 | key=chennai.adyar.ride | id=5b3bb046
GOOD MESSAGE #3 | key=bangalore.indiranagar.food | id=3e1d58c0
GOOD MESSAGE #4 | key=bangalore.koramangala.food | id=e0609a71
GOOD MESSAGE #5 | key=chennai.annanagar.food | id=4403f1d4
GOOD MESSAGE #6 | key=bangalore.indiranagar.ride | id=f513c4a8
GOOD MESSAGE #7 | key=chennai.adyar.food | id=49d127ae
GOOD MESSAGE #8 | key=chennai.adyar.food | id=70eda139
GOOD MESSAGE #9 | key=bangalore.indiranagar.ride | id=31a89177
BAD MESSAGE #10 | key=chennai.annanagar.food | missing zone | id=e1cfe50d
GOOD MESSAGE #11 | key=chennai.tnagar.food | id=f2e6da6f
GOOD MESSAGE #12 | key=bangalore.whitefield.ride | id=da70b012
GOOD MESSAGE #13 | key=chennai.adyar.food | id=92e7c837
GOOD MESSAGE #14 | key=bangalore.whitefield.food | id=38584d6b
GOOD MESSAGE #15 | key=chennai.adyar.food | id=36fda635
GOOD MESSAGE #16 | key=chennai.adyar.ride | id=d15fee1e
GOOD MESSAGE #17 | ke

In [20]:
!grep -h "PROCESSED" *_d3.log | head -15

PROCESSED by Arjun | key=bangalore.indiranagar.ride | message={'id': '6c17c7f0', 'city': 'bangalore', 'zone': 'indiranagar', 'type': 'ride', 'ts': '2026-11-16T13:54:38.045152+00:00'}
PROCESSED by Arjun | key=bangalore.whitefield.ride | message={'id': 'a4e358a2', 'city': 'bangalore', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-16T13:54:46.052242+00:00'}
PROCESSED by Arjun | key=bangalore.whitefield.ride | message={'id': '84ef9938', 'city': 'bangalore', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-16T13:54:48.053978+00:00'}
PROCESSED by Arjun | key=bangalore.whitefield.ride | message={'id': 'e9d613d9', 'city': 'bangalore', 'zone': 'whitefield', 'type': 'ride', 'ts': '2026-11-16T13:54:56.060039+00:00'}
PROCESSED by Arjun | key=bangalore.koramangala.ride | message={'id': 'cb2cda16', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'ride', 'ts': '2026-11-16T13:55:00.061400+00:00'}
PROCESSED by Arjun | key=bangalore.whitefield.ride | message={'id': 'b471ec35', 'city': 'ba

In [21]:
!grep -h "REJECTED" *_d3.log

REJECTED by Arjun | key=bangalore.indiranagar.ride | reason=missing zone | id=f30d56f8
REJECTED by Arjun | key=bangalore.whitefield.ride | reason=missing zone | id=961c32c4
REJECTED by Control | key=bangalore.indiranagar.ride | reason=missing zone | id=f30d56f8
REJECTED by Control | key=bangalore.koramangala.food | reason=missing zone | id=ffcb369d
REJECTED by Control | key=bangalore.whitefield.ride | reason=missing zone | id=961c32c4
REJECTED by Meena | key=bangalore.koramangala.food | reason=missing zone | id=ffcb369d
REJECTED by Priya | key=bangalore.indiranagar.ride | reason=missing zone | id=f30d56f8
REJECTED by Ravi | key=bangalore.koramangala.food | reason=missing zone | id=ffcb369d


In [22]:
!echo "Good messages processed:"
!grep -h -c "PROCESSED" *_d3.log

!echo "Bad messages rejected:"
!grep -h -c "REJECTED" *_d3.log

Good messages processed:
66
112
0
15
46
43
28
Bad messages rejected:
2
3
0
0
1
1
1


In [23]:
!cd /content && zip -r stream.zip stream

  adding: stream/ (stored 0%)
  adding: stream/ravi.log (deflated 90%)
  adding: stream/control.log (deflated 90%)
  adding: stream/karthik.log (deflated 90%)
  adding: stream/driver.py (deflated 65%)
  adding: stream/priya.log (deflated 90%)
  adding: stream/control_d3.log (deflated 90%)
  adding: stream/meena_d3.log (deflated 90%)
  adding: stream/meena.log (deflated 90%)
  adding: stream/karthik_d3.log (deflated 89%)
  adding: stream/arjun.log (deflated 90%)
  adding: stream/priya_d3.log (deflated 90%)
  adding: stream/ravi_d3.log (deflated 90%)
  adding: stream/arjun_d3.log (deflated 90%)
  adding: stream/dispatcher_d3.log (deflated 83%)
  adding: stream/dispatcher.py (deflated 62%)
  adding: stream/dispatcher.log (deflated 89%)


In [24]:
from google.colab import files
files.download('/content/stream.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>